# 你的第一个实验室（社区贡献：市场调研智能体）

### 请先读完本节。篇幅虽长，但对起步非常有价值。

## 练习目标

在几分钟内搭出一个「有用的 LLM 小方案」：给一个 URL，抓取网页正文，再让 Frontier 模型返回摘要——互联网版「读者文摘」。

本笔记本后半段会把同一套抓取 + `messages` 流水线升级成**市场调研顾问**风格的推荐智能体（SWOT / Porter 等框架写在 system prompt 里）。

课程后期你会做到多代理协作；今天先把「请求网页 → 清洗文本 → Chat Completions」跑通。

## 开始前

请先完成 [PC](../SETUP-PC.md) 或 [Mac](../SETUP-mac.md) 环境配置；建议从项目根目录启动 Jupyter，并激活课程虚拟环境。

## 如果您是 Jupyter Lab 新手

单击代码单元格，按 **Shift+Enter** 执行。可用工具栏 `+` 增删单元格、打印中间变量、做小实验。

也可参考 [Jupyter 指南](Guide%20to%20Jupyter.ipynb)：Markdown、用 `!` 跑 shell、`tqdm` 进度条等。

## 如果您是命令行新手

参考：[PC 上的命令行](https://chatgpt.com/share/67b0acea-ba38-8012-9c34-7a2541052665) 与 [Mac 上的命令行](https://chatgpt.com/canvas/shared/67b0b10c93a081918210723867525d2b)。

## 如果您更喜欢在 IDE 里工作

VS Code / Cursor / PyCharm 都能跑这些实验室笔记本。VS Code 配置可参考：[课程 IDE 配置说明](https://chatgpt.com/share/676f2e19-c228-8012-9911-6ca42f8ed766)。

## 如果想温习 Python

可看 [Intermediate Python](Intermediate%20Python.ipynb)。若你已熟悉类似写法，可跳过：

```python
# 示意：从字典安全取字段
book.get("author")
```

## 我是来帮忙的

有问题可通过课程平台联系，或发邮件到 ed@edwarddonner.com，也可 LinkedIn：https://www.linkedin.com/in/eddonner/  
（作者也在尝试 X/Twitter：[@edwarddonner](https://x.com/edwarddonner)）

## 更多疑难排查

见同目录 [troubleshooting](troubleshooting.ipynb)。文末有诊断脚本。

## 基础技术知识（Git、API、调试）

相对新手也欢迎：必备的是耐心。自学指南（Git/GitHub、API、Python、调试）见：

- Git 与 GitHub：https://github.com/ed-donner/agents/blob/main/guides/03_git_and_github.ipynb
- 技术基础（环境变量、端点等）：https://github.com/ed-donner/agents/blob/main/guides/04_technical_foundations.ipynb
- Python 基础：https://github.com/ed-donner/agents/blob/main/guides/06_python_foundations.ipynb
- 调试：https://github.com/ed-donner/agents/blob/main/guides/08_debugging.ipynb

同文件夹还有更多指南，多数与 LLM 工程相关。

## 如果这些对你已是旧帽子

仍建议快速过一遍前几格实验——后面几周会明显加深，最终会微调自己的模型去对齐 OpenAI 能力。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读 — 重要说明</h2>
            <span style="color:#900;">本课的节奏可能和你上过的其他课不同：讲师不会在你盯着屏幕时逐字敲代码，而是带着跑 Jupyter Lab，帮你建立直觉。建议你<strong>看完讲座之后</strong>亲自从上到下执行本笔记本：加 <code>print</code>、改 URL、提交自己的变体到 GitHub——既是练习，也是作品集。</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">此代码是实时资源 — 请留意公告邮件</h2>
            <span style="color:#f71;">仓库会定期更新示例与注释；笔记本可能与视频略有出入，但视频里的核心都会保留，并补充更好的说明与新模型。可把它当成互动书。<br/><br/>
                重要更新常见于 Udemy 左侧「公告」。也可在通知设置里选择接收邮件。</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">这些练习的商业价值</h2>
            <span style="color:#181;">笔记本会穿插有趣实验，但目标始终是可落地的商业技能：摘要、调研、推荐……学每个 API 与 prompt 技巧时，想一想能否迁到你的业务场景。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 导入：网页抓取 + OpenAI 所需依赖 ==========

# 标准库 os：读环境变量（例如 OPENAI_API_KEY）
import os
# 第三方 requests：用 HTTP GET 拉取网页 HTML
import requests
# dotenv.load_dotenv：把 .env 中的密钥载入环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# BeautifulSoup：解析 HTML，抽出标题与正文
from bs4 import BeautifulSoup
# IPython 展示：把模型返回的 Markdown 漂亮地渲染出来
from IPython.display import Markdown, display
# OpenAI 官方 SDK 客户端
from openai import OpenAI

# 若本格 import 失败：先查同目录 troubleshooting 笔记本（依赖/环境问题居多）


# 连接到 OpenAI（或 Ollama）

下一格会加载 `.env` 里的环境变量，并准备连接 OpenAI。

若想用免费的本地 **Ollama**，请看 README「付费 API 的免费替代方案」；完整示例见解决方案文件夹的 `day1_with_ollama.ipynb`。

## 遇到问题怎么排

打开同目录 [troubleshooting](troubleshooting.ipynb)，按步骤定位根因。

改过代码仍异常时：菜单 **内核 → 重新启动内核并清除所有输出**，再从顶部重新跑。

也可联系课程作者：ed@edwarddonner.com。

API 成本通常很低且可控；也可用 Ollama 作免费替代（第 2 天会展开）。


In [ ]:
# ========== 加载 .env，并对 API Key 做三道体检 ==========

# override=True：允许 .env 覆盖进程里已有的同名变量（保持原参数）
load_dotenv(override=True)
# 读取 OpenAI 密钥；环境变量名必须是 OPENAI_API_KEY
api_key = os.getenv('OPENAI_API_KEY')

# 检查钥匙：以下 print 文案是排错指引的一部分，保持英文原文

# 情况 1：完全没读到 key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 情况 2：有值但不像项目密钥前缀 sk-proj-
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 情况 3：首尾可能粘了空格/制表符
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 通过基础检查
    print("API key found and looks good so far!")


In [ ]:
# ========== 创建 OpenAI 客户端（默认从环境变量取密钥） ==========

# OpenAI()：无参时通常读取环境里的 OPENAI_API_KEY
openai = OpenAI()

# 若本格失败：内核 → 重新启动并清除输出，再从头跑
# 仍失败：打开同目录 troubleshooting 笔记本按步骤排查


# 快速预览：先对 Frontier 模型做一次最小调用

确认客户端可用后，再进入「抓网页 → 摘要」主线。


In [ ]:
# ========== 最小 Chat Completions 预览 ==========

# 用户消息原文（prompt 字符串勿翻译，以免改变模型行为）
message = "Hello, GPT! This is my first ever message to you! Hi!"
# 调用 chat.completions：指定模型 gpt-4o-mini，messages 仅含一条 user
response = openai.chat.completions.create(model="gpt-4o-mini", messages=[{"role":"user", "content":message}])
# 打印第一条候选回复的文本
print(response.choices[0].message.content)


## 开始我们的第一个项目

下面用类封装「URL → 标题 + 纯文本」，为后续摘要打地基。


In [ ]:
# ========== Website 类：用 requests + BeautifulSoup 抓取并清洗正文 ==========

# 若不熟悉 class，可先看 Intermediate Python 笔记本

# 有些站点会校验 User-Agent；伪装成常见浏览器可降低被拒概率
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        # 保存原始 URL，便于调试
        self.url = url
        # GET 网页；headers 带上上面的浏览器标识
        response = requests.get(url, headers=headers)
        # 用 html.parser 解析响应字节内容
        soup = BeautifulSoup(response.content, 'html.parser')
        # 取 <title>；没有则用占位英文串（字符串保持原文）
        self.title = soup.title.string if soup.title else "No title found"
        # 去掉 script/style/img/input 等对摘要无用的节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 从 body 抽出纯文本：换行分隔，并 strip 空白
        self.text = soup.body.get_text(separator="\n", strip=True)


In [ ]:
# ========== 试抓课程作者主页：看 title 与 text ==========

# 构造 Website；可改 URL 做实验（改了就不是原实验对象了）
ed = Website("https://edwarddonner.com")
# 打印标题
print(ed.title)
# 打印清洗后的正文（可能很长）
print(ed.text)


In [ ]:
# ========== 再试一个 GitHub 主页（贡献者示例） ==========

# 变量名 rudra、URL 均保持原样
rudra=Website("https://github.com/RudraDudhat2509/")
print(rudra.title)
print(rudra.text)


## 提示类型（Prompt Types）

大模型（如 GPT-4o）被训练成按固定角色接收指令，常见两类：

- **系统提示（system）**：任务是什么、语气/格式约束
- **用户提示（user）**：本轮要处理的具体内容（对话起点）

后面会把「网站正文」塞进 user，把「如何摘要」写进 system。


In [ ]:
# ========== 定义 system_prompt：约束摘要风格（字符串保持英文原文） ==========

# 可自行实验改最后一句语气；但默认字符串请先跑通再改
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown. Always use Points and simple english. Never use hyphens. Stick to the point"


In [ ]:
# ========== user_prompt_for：把 Website 对象编成用户侧提示 ==========

def user_prompt_for(website):
    # 先写入标题，让模型知道页面主题
    user_prompt = f"You are looking at a website titled {website.title}"
    # 再追加任务说明（英文 prompt 原文勿改）
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    # 最后附上抓取到的正文
    user_prompt += website.text
    return user_prompt


In [ ]:
# ========== 预览：打印 ed 站点对应的完整 user prompt ==========

print(user_prompt_for(ed))


## 消息（messages）结构

OpenAI Chat Completions（以及许多兼容 API）期望这样的列表：

```python
[
    {"role": "system", "content": "系统消息写这里"},
    {"role": "user", "content": "用户消息写这里"}
]
```

接下来两格先做一次「与网站无关」的玩具调用，熟悉 `role` 字段；之后再接到真正的摘要流水线。


In [ ]:
# ========== 玩具 messages：system 定人设，user 问算术 ==========

# content 字符串保持英文原文（影响模型回复风格）
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]


In [ ]:
# ========== 用上面的 messages 调用 gpt-4o-mini 并打印 ==========

response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(response.choices[0].message.content)


## 用函数为 GPT-4o-mini 组装「网站摘要」messages

把 system_prompt 与 `user_prompt_for(website)` 打包成 API 需要的列表。


In [ ]:
# ========== messages_for：返回 [system, user] 两条消息 ==========

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [ ]:
# ========== 试调用：看 ed 对应的 messages 结构 ==========

messages_for(ed)


## 串起来：OpenAI API 调用其实很短

抓取 → 组 messages → `chat.completions.create` → 取 `content`。


In [ ]:
# ========== summarize(url)：端到端返回摘要字符串 ==========

def summarize(url):
    # 1) URL → Website（标题+正文）
    website = Website(url)
    # 2) 调 Chat Completions；模型名保持 gpt-4o-mini
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    # 3) 返回第一条回复文本
    return response.choices[0].message.content


In [ ]:
# ========== 对作者主页跑一遍 summarize ==========

summarize("https://edwarddonner.com")


In [ ]:
# ========== display_summary：摘要结果用 Markdown 渲染 ==========

def display_summary(url):
    # 先拿到纯文本摘要
    summary = summarize(url)
    # 在 Jupyter 里按 Markdown 显示（标题/列表更易读）
    display(Markdown(summary))


In [ ]:
# ========== 渲染展示：edwarddonner.com ==========

display_summary("https://edwarddonner.com")


# 再试更多网站

注意：这种「静态 HTML + BeautifulSoup」只适合可直接抓到正文的站点。

用 JavaScript 渲染的页面（如很多 React 应用）可能抓不到实质内容——社区贡献里常有 Selenium 方案；安装方式可自行查文档或问 ChatGPT。

受 CloudFront 等保护的站点可能返回 403（感谢 Andy J 指出）。

但许多站点用本方法已经够用。


In [ ]:
# ========== 试摘要新闻站 CNN ==========

display_summary("https://cnn.com")


In [ ]:
# ========== 试摘要贡献者 GitHub 主页 ==========

display_summary("https://github.com/RudraDudhat2509")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">你刚刚体验了调用 Frontier 模型的云端 API。除了后续可能自建/微调模型，课程许多阶段都会用到 OpenAI 这类 API。<br/><br/>
            这里的用例是<strong>摘要</strong>——经典 GenAI 场景：新闻、财报、简历/求职信……想想如何接到你的业务，并快速做个原型。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前 — 现在就试你的变体</h2>
            <span style="color:#900;">用下面单元格做自己的简单商业示例。仍可围绕摘要：例如根据邮件正文建议简短主题行——正是商务邮件工具常见能力。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 升级：市场调研顾问式 system + recommend(url) ==========

# 第 1 步：创建提示（system_prompt 全文保持英文原文，勿翻译）

system_prompt = """You are to act like a Mckinsey Consultant specializing in market research. 
1) You are to follow legal guidelines and never give immoral advice. 
2) Your job is to maximise profits for your clients by analysing their companies initiatives and giving out recommendations for newer initiatives.\n 
3) Follow industry frameworks for reponses always give simple answers and stick to the point.
4) If possible try to see what competitors exist and what market gap can your clients company exploit.
5) Further more, USe SWOT, Porters 5 forces to summarize your recommendations, Give confidence score with every recommendations
6) Try to give unique solutions by seeing what the market gap is, if market gap is ambiguious skip this step
7) add an estimate of what rate the revenue of the comapany will increase at provided they follow the guidelines, give conservating estimates keeping in account non ideal conditions.
8) if the website isnt of a company or data isnt available, give out an error message along the lines of more data required for analysis"""

def makereq(url):
  # 抓取目标公司/产品站
  website=Website(url)
  # 组装 user 侧英文请求（字符串模板保持原文）
  user_prompt=f"This is my companies website: {website.title}. Could you help me increase profits by giving me recommendations on what i should do. here is the content of my website:\n"
  # 把网页正文接到提示后面
  user_prompt+=website.text;
  # 返回 Chat Completions 所需的 messages 列表
  return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
  ]
def recommend(url):
        # 调用 gpt-4o-mini；messages 由 makereq 现场生成
        response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = makereq(url))
        # 把顾问回复按 Markdown 渲染到笔记本输出区
        display(Markdown(response.choices[0].message.content))
        



In [ ]:
# ========== 试 recommend：Swiggy 企业站 ==========

recommend("https://www.swiggy.com/corporate/")


In [ ]:
# ========== 试 recommend：Valorant 官网 ==========

recommend("https://playvalorant.com/en-us/")


In [ ]:
# ========== 试 recommend：Nexora Labs ==========

recommend("https://nexora-labs.com/")


In [ ]:
# ========== 试 recommend：GitHub 个人页（可能触发「数据不足」类回复） ==========

recommend("https://github.com/RudraDudhat2509/")
